# Chapter 16. Transformers and Autoregressive Language Models

## 16.2 Learning Objectives

By the end of this chapter, you should be able to:

- describe the main components of a Transformer block;
- explain the roles of queries, keys, and values in self-attention;
- implement a simple autoregressive generation loop;
- describe the architecture of a decoder-only Transformer;
- explain the purpose and memory cost of a KV cache.

## 16.3 Inside a Transformer

In Chapter 8, we treated the Transformer mostly as a pretrained model that turned tokens into contextual representations. Now we'll look more closely at what happens inside it. A Transformer is built by stacking a sequence of similar blocks.

A modern block typically contains two major transformations:
1. **self-attention**;
2. a **feed-forward network**, sometimes called an MLP.

Residual connections allow the input of each transformation to bypass it and be added back to its output. Normalization layers stabilize the representations as they pass through the network. The exact ordering and implementation vary across architectures.

Modern language models also commonly use variants such as:
- RMSNorm instead of LayerNorm;
- SiLU-based gated feed-forward networks;
- grouped-query attention;
- Rotary Position Embeddings.

We'll use **SmolLM2-135M**, a compact decoder-only language model from Hugging Face. It contains approximately 135 million parameters, making it small enough to inspect and run comfortably in a free Colab environment while still using many of the architectural ideas found in considerably larger modern LLMs. Most importantly, we'll use the **base pretrained model**, not its instruction-tuned counterpart.

The distinction will become important later. A base language model learns to predict and continue text. An instruction-tuned model receives additional training that encourages it to respond to instructions and participate in structured conversations. We'll explore that distinction properly in the next chapter.

SmolLM2-135M is a Transformer decoder model trained on approximately 2 trillion tokens. Its configuration uses a context length of 8,192 positions, Rotary Position Embeddings, and grouped-query attention.

## 16.4 Self-Attention

Consider this sentence:

> The animal didn't cross the street because **it** was tired.

What does `it` refer to? To build a useful representation of that token, the model needs information from other tokens in the sequence. The purpose of self-attention is to let each token gather information from other tokens. In Chapter 15, we started with input embeddings and had an encoder-based Transformer produce contextual embeddings. Self-attention is one of the mechanisms that turns the first representation into the second.

### 16.4.1 Queries, Keys, and Values

Self-attention starts from the hidden representation of each token. Suppose our sequence is represented by a matrix, X, where each row corresponds to one token. The model applies three learned linear transformations:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

The very same input, X, produces three new representations:

- **Query**, (Q);
- **Key**, (K);
- **Value**, (V).

Why three? An intuitive way to think about them is:
- a **query** describes what information a token is looking for;
- a **key** describes what information a token may provide;
- a **value** contains the information that can actually be gathered.

These are only intuitions—the vectors themselves are learned numerical representations. For every query, the model compares it with the keys of the tokens it is allowed to attend to.

### 16.4.2 Attention Scores

The similarity between a query and a key is computed using a dot product:

$$
QK^T
$$

If we have four tokens, the resulting matrix contains one score for every possible pair, so it is 4x4 in shape. Each row represents the token receiving information. Each column represents a token it may receive information from. The scores are divided by the square root of the key dimension:

$$
\frac{QK^T}{\sqrt{d_k}}
$$

This scaling prevents the magnitude of dot products from growing too large as the dimensionality increases. Then a softmax transforms the scores into weights:

$$
A =
\mathrm{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
$$

Finally, those weights are used to combine the values:

$$
A V
$$

Putting everything together:

$$
\mathrm{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

This is **scaled dot-product attention**. The result for each token is a weighted combination of information provided by the other tokens.

## 16.4.3 A Small Example

Let's create a tiny sequence representation:

In [4]:
import torch
import torch.nn as nn

torch.manual_seed(42)

batch_size = 1
seq_len = 4
hidden_size = 8

x = torch.randn(batch_size, seq_len, hidden_size)
x.shape

torch.Size([1, 4, 8])

We have a dummy batch of one sequence with four tokens, each token being represented by a vector of dimension eight. Now, let's create three learned projections, for queries, keys, and values. Notice that these are simply regular linear layers, there's nothing special about them.

In [5]:
q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
v_proj = nn.Linear(hidden_size, hidden_size, bias=False)

By feeding our dummy batch to these linear layers, we obtain three different representations that we're calling queries, keys, and values:

In [8]:
q = q_proj(x)
k = k_proj(x)
v = v_proj(x)
q.shape, k.shape, v.shape

(torch.Size([1, 4, 8]), torch.Size([1, 4, 8]), torch.Size([1, 4, 8]))

As you can see, they all have exactly the same shape. What changed is not their dimensions, but their roles in the attention mechanism. For every token, the query represents what that token is looking for, the key represents what it has to offer for matching purposes, and the value contains the information that may eventually be passed forward.

The next step is to compare every query against every key. We can do that by taking their dot products. Since `q` has a shape of (1, 4, 8), we transpose the last two dimensions of `k`, turning its shape from (1, 4, 8) into (1, 8, 4). Matrix multiplication then produces a (1, 4, 4) tensor: one score for every possible pair of tokens.

There is one additional detail, though. Dot products tend to grow in magnitude as the dimensionality of the vectors increases. Large values may push the softmax function toward very sharp distributions, which can make optimization more difficult. Scaled dot-product attention addresses this by dividing the dot products by the square root of the query/key dimension, which is eight in our small example.

In [10]:
import math
dotprod = q @ k.transpose(-2, -1) 
scores = dotprod / math.sqrt(hidden_size)
scores, scores.shape

(tensor([[[ 0.4428,  0.0150, -0.1450,  0.2133],
          [-0.2337,  0.0387, -0.0665,  0.2461],
          [ 0.1223, -0.2116, -0.2011, -0.8662],
          [-0.1039,  0.3430,  0.6562,  0.0110]]], grad_fn=<DivBackward0>),
 torch.Size([1, 4, 4]))

Each row in this matrix corresponds to one query token, while each column corresponds to one key token. For example, the value at position [0, 2, 3] measures how well the query representation of the third token matches the key representation of the fourth token.

These scores can be positive or negative and, by themselves, they are not yet attention weights. We would like each token to distribute its attention across the entire sequence in a way that is easier to interpret and use. For that, we apply a softmax along the last dimension.

In [11]:
weights = torch.softmax(scores, dim=-1)
weights

tensor([[[0.3331, 0.2171, 0.1850, 0.2648],
         [0.1957, 0.2569, 0.2313, 0.3161],
         [0.3556, 0.2547, 0.2574, 0.1323],
         [0.1717, 0.2685, 0.3672, 0.1926]]], grad_fn=<SoftmaxBackward0>)

Now every row contains positive values that add up to one. These are the actual attention weights.

For instance, the first token assigns roughly 33% of its attention to itself, 22% to the second token, 19% to the third, and 26% to the fourth. Remember that these values are completely arbitrary in our example because the projection layers were randomly initialized. In a trained Transformer, the query and key projections are learned so that useful relationships between tokens receive larger attention weights.

There is still one piece missing. So far, queries and keys have told us where to look, but they have not determined the information that should be returned. That's the role of the values.

We multiply the attention weights by the value vectors so that every token receives a weighted combination of the information carried by all tokens in the sequence.

In [12]:
context = weights @ v
context, context.shape

(tensor([[[ 0.7068, -0.1789,  0.0215,  0.4855,  0.6577, -0.4675,  0.1549,
            0.0205],
          [ 0.6226, -0.2538, -0.1157,  0.3893,  0.6098, -0.5600,  0.3241,
            0.0212],
          [ 0.7434, -0.1621,  0.0762,  0.4636,  0.6509, -0.5292,  0.0863,
            0.0480],
          [ 0.6316, -0.2643, -0.1571,  0.3258,  0.5874, -0.7131,  0.2960,
            0.0566]]], grad_fn=<UnsafeViewBackward0>),
 torch.Size([1, 4, 8]))

And we're back to the original shape: one vector of dimension eight for each of the four tokens.

The important difference is that these vectors are no longer based only on the corresponding token. Each output vector is a weighted combination of the value vectors from the entire sequence. In other words, attention has allowed every token representation to incorporate information from the other tokens.

We can summarize the complete operation as:

$$
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

where ($d_k$) is the dimensionality of the query and key vectors. This is called scaled dot-product attention.

We implemented every step separately so we could see exactly what is going on. In practice, however, PyTorch already provides an implementation of this operation.

## 16.5 Attention in PyTorch

PyTorch provides scaled dot-product attention through the `scaled_dot_product_attention()` function, usually referred to simply as SDPA.

Conceptually, it performs the same steps we just implemented manually: it computes the query-key dot products, scales them, applies the attention weights, and combines the value vectors. Besides saving us from writing these operations ourselves, SDPA gives PyTorch the opportunity to use optimized attention implementations when the hardware, data type, and tensor shapes allow it.

The function is also a lower-level building block rather than a complete Transformer layer. We still create the query, key, and value projections ourselves, and later we'll use it to construct multi-head attention.

Before calling SDPA, let's slightly reshape our tensors.

In [ ]:
import torch.nn.functional as F

q = q.unsqueeze(1)
k = k.unsqueeze(1)
v = v.unsqueeze(1)

q.shape

torch.Size([1, 1, 4, 8])

We've inserted an extra dimension between the batch and sequence dimensions. Our tensors now follow the shape:

$$
(\text{batch},\ \text{heads},\ \text{sequence},\ \text{head dimension})
$$

In our example, that is (1, 1, 4, 8): one sequence in the batch, one attention head, four tokens, and eight features per head.

We only have one attention head for now, so this new dimension has size one. Once we move to multi-head attention, this dimension will contain several heads operating in parallel.

Now we can pass the three tensors directly to SDPA. We set `dropout_p=0.0` because we are only demonstrating the operation and do not want dropout to alter the result.

In [ ]:
context_sdpa = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
context_sdpa.shape

torch.Size([1, 1, 4, 8])

The output has the same shape as the query tensor: (1, 1, 4, 8). If we remove the temporary head dimension, we recover the same shape as the context tensor we computed manually.

In [ ]:
torch.testing.assert_close(context_sdpa.squeeze(1), context)

No exception is raised, so both implementations produce the same result within normal numerical tolerance.

This is useful because the concise SDPA call is not introducing a different kind of attention. It is implementing the same computation we just built step by step. The difference is that PyTorch can choose an efficient implementation for the operation depending on the execution environment.

Instead of manually computing attention using the dot-product, we can perform a single call to SDPA. It can also accept an attention mask to control which token pairs are allowed to participate in attention. This will become particularly important when we turn our Transformer into an autoregressive language model. But first, let's expand attention beyond a single head.

## 16.6 Multi-Head Attention

So far, we've performed one attention operation over the entire hidden representation. Transformers normally use several attention operations in parallel. These are called **attention heads**. So far, we used a single attention head that took the full length of each embeding in its computations. What if we divide the full vector among six heads instead? For an embedding of dimensionality 384, if we split it among six heads, each head will handle 64 dimensions only. Each head gets its own chunk of query, key, and value representations.
The heads perform attention independently, their results are concatenated, and another learned projection combines them.

### 16.6.1 Why Multiple Heads?

Different heads give the model multiple representational subspaces in which to compare tokens. We should be careful not to overinterpret them. It's tempting to say something along the lines of "*This head learns syntax, while that one learns names, and another one learns pronouns.*" Individual heads sometimes exhibit recognizable patterns, but there is no guarantee that each head corresponds neatly to a human-interpretable concept. The important point is actually much simpler: multiple heads allow several different attention patterns to be represented simultaneously.

A multi-head attention tensor commonly has a shape such as (N, H, L, D_head) instead of the typical (N, L, D) we're used to. It's also important to keep in mind that H * D_head must match D exactly, there can be no remainder. For example, if we had a mini-batch of four sequences, each having 100 tokens, and each token being represented by an embedding with 384 dimensions, we could have a model with six heads (since 384 divided by six is 64) but not ten heads (since there is a remainder of four). The (N, H, L, D_head) representation is the one expected by SDPA.

### 16.6.2 Grouped-Query Attention

Many modern LLMs use a variation called **Grouped-Query Attention**, or GQA. Instead of using the same number of heads for each and every projection (queries, keys, and values), the number of heads used for keys and values is typically reduced, so several query heads share one set of key and value heads. Why, you ask? Keys and values from previous tokens are the representations that need to be stored during autoregressive generation. Reducing the number of key/value heads therefore reduces the memory required by the KV cache. PyTorch's SDPA API directly supports grouped-query attention through its `enable_gqa` argument.

The small model we'll use later provides a concrete example. Its configuration contains nine query heads and only three key and value heads, so each group of three query heads shares one key/value head. We'll return to the significance of key/value storage when we discuss autoregressive inference.

## 16.7 Position in a Sequence

Attention has an interesting limitation. By itself, the operation does not know whether a token came first, second, or hundredth in the sequence. Two sentences containing the same tokens would be indistinguishable under a naive implementation of self-attention, even if their order made their meanings completely different. For example, the sentences "_dog bites man_" and "_man bites dog_" both have exactly the same three tokens, but they are far from conveying the same idea. A language model therefore needs some way to represent **position**.

One solution is to associate every position with an embedding, thus adding a positional embedding to the input embedding we discussed earlier. In this case, the first token receives the embedding for position `0`, the second receives position `1`, and so on. This is the approach we encountered indirectly with RoBERTa in Chapter 8, as its embedding module contains learned positional embeddings. The model therefore learns a vector associated with each supported position. This works, but many modern decoder-only models use a different approach.

## 16.8 Rotary Position Embeddings

Our SmolLM2 model uses **Rotary Position Embeddings**, or **RoPE**. RoPE does not simply add a learned positional vector to the token representation. Instead, it modifies the **query and key vectors used by attention**. The basic idea is surprisingly geometric.

Consider a query vector with two dimensions only, for the sake of simplicity:

$$
(x_1, x_2)
$$

A two-dimensional vector can be rotated by angle ($\theta$) using:

$$
\begin{bmatrix}
\cos\theta & -\sin\theta 
\\
\sin\theta & \cos\theta
\end{bmatrix}
\begin{bmatrix}
x_1
\\
x_2
\end{bmatrix}
$$

RoPE groups dimensions into pairs and rotates them using angles determined by:
- the token's position;
- a set of frequencies.

Different positions therefore receive different rotations.

### 16.8.2 Applying RoPE to Attention

Suppose two identical token representations occur at positions 5 and 20. Their query and key vectors receive different rotations because their positions differ. The dot products used by attention therefore contain information about the relative positions of the tokens. This gives the model positional awareness without adding an explicit positional vector to every hidden state. An important detail is that RoPE is normally applied to queries and keys, but not values, because position needs to affect the attention relationship between queries and keys. Hugging Face describes RoPE as rotating query and key vectors according to their positions so that their interaction encodes relative positional information.

### 16.8.3 RoPE Frequencies

Different pairs of vector dimensions rotate at different frequencies. Some change rapidly with position, while others change much more slowly. This gives the attention mechanism positional information across different scales. A configuration parameter commonly called "theta" controls the base frequency.

Let's inspect the configuration of the model we'll use without loading its weights yet:

In [3]:
from transformers import AutoConfig

repo_id = "HuggingFaceTB/SmolLM2-135M"

config = AutoConfig.from_pretrained(repo_id)
config.rope_parameters['rope_theta'], config.max_position_embeddings

(100000, 8192)

Modern model families also use scaled RoPE variants designed to extend useful context lengths, including linear, dynamic, YaRN, LongRoPE, and model-specific variants. We won't explore those techniques here, but they all build on the same underlying idea of encoding position through rotations.

### 16.8.4 Why RoPE Matters

For our purposes, remember three things:
1. attention alone does not encode token order;
2. RoPE injects positional information by rotating query and key vectors;
3. many contemporary decoder-only LLMs use RoPE or a variation of it.

We now have contextual attention **and** positional information. But there's still one major problem before we can train a language model. Our attention can currently look everywhere, that is, it can cheat.

## 16.9 From Bidirectional to Causal Attention

The encoder model we used in Chapter 8 could process a sentence as a whole. In order to generate contextual representations of every token, it needs to be aware of the full context, that is, an encoder may use information from tokens both before and after it. That makes sense for representation tasks, when the entire input is already known. But generation is different. If we're training a model to predict the next word, at any given token, the model should not be able to "see" which tokens follow it.

### 16.9.1 The Causal Constraint

During causal language modeling, the token at position (t) may attend only to positions up to and including (t) itself. It cannot attend to "future" tokens, that is, the tokens that follow it. The first token may see itself only, but the second token may see itself and the first one, and so on, and so forth. This restriction is enforced using a **causal attention mask**, which excludes attention scores (makes them zero) for "future" tokens.

In PyTorch, we can use `is_causal=True` to prevent each query position from attending to future key positions. This seemingly small change transforms attention into something suitable for autoregressive language modeling.

In [ ]:
causal_context = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)

## 16.10 The Decoder-Only Transformer

### 16.10.1 Loading SmolLM2

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")

tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForCausalLM.from_pretrained(repo_id)
model = model.to(device)
model.eval()

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

In [5]:
model.config.hidden_size, model.config.num_attention_heads, model.config.num_key_value_heads

(576, 9, 3)

In [6]:
model.model.layers[0]

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=576, out_features=576, bias=False)
    (k_proj): Linear(in_features=576, out_features=192, bias=False)
    (v_proj): Linear(in_features=576, out_features=192, bias=False)
    (o_proj): Linear(in_features=576, out_features=576, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
    (up_proj): Linear(in_features=576, out_features=1536, bias=False)
    (down_proj): Linear(in_features=1536, out_features=576, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
)

In [ ]:
model.lm_head

Linear(in_features=576, out_features=49152, bias=False)

In [ ]:
model.config.vocab_size

49152

## 16.11 Causal Language Modeling

In [7]:
prompt = "The capital of Spain is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
inputs["input_ids"]

tensor([[ 504, 3575,  282, 7476,  314]])

In [8]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())

['The', 'Ġcapital', 'Ġof', 'ĠSpain', 'Ġis']

In [11]:
with torch.inference_mode():
    outputs = model(**inputs)

outputs.logits.shape

torch.Size([1, 5, 49152])

### 16.11.1 The Next Token

In [12]:
next_token_logits = outputs.logits[:, -1, :]
next_token_logits.shape

torch.Size([1, 49152])

In [13]:
next_token_probs = torch.softmax(next_token_logits, dim=-1)
top_probs, top_ids = torch.topk(next_token_probs, k=5, dim=-1)
top_probs, top_ids

(tensor([[0.2393, 0.2109, 0.0442, 0.0344, 0.0269]], dtype=torch.bfloat16),
 tensor([[28030,   260,  3807,  1217, 30010]]))

In [14]:
for probability, token_id in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode([token_id])

    print(f"{token!r}: " f"{probability:.4f}")

' Madrid': 0.2393
' the': 0.2109
' located': 0.0442
' called': 0.0344
' Barcelona': 0.0269


## 16.12 Training with Next-Token Prediction

How was the model trained to produce these logits? By repeatedly solving the same task we're performing now, that is, predicting the next token. Suppose the training sequence is "*The weather is very nice today*." Starting with this short sequence, we can produce many training examples, as each position predicts the token immediately to its right. More explicitly:

> "The"                    → "weather"
> 
> "The weather"            → "is"
> 
> "The weather is"         → "very"
> 
> "The weather is very"    → "nice"
> 
> "The weather is very nice" → "today"

Because causal attention prevents "future" tokens from being visible, all of these predictions can be trained in parallel. This is one reason causal language-model pretraining can learn so much from text. A sequence of (N) tokens provides approximately (N-1) next-token training targets. For a sequence containing 1,000 tokens, we obtain nearly 1,000 prediction tasks from that one sequence alone.

### 16.12.1 Shifted Labels

In [16]:
training_text = "The weather is very nice today"
training_inputs = tokenizer(training_text, return_tensors="pt")
input_ids = training_inputs["input_ids"]
input_ids

tensor([[ 504, 3947,  314, 1035, 9239, 1834]])

In [17]:
# inputs
input_ids[:, :-1]

tensor([[ 504, 3947,  314, 1035, 9239]])

In [ ]:
# labels
input_ids[:, 1:]

tensor([[3947,  314, 1035, 9239, 1834]])

### 16.12.2 Cross-Entropy Loss

In [18]:
training_inputs = {key: value.to(device) for key, value in training_inputs.items()}

with torch.inference_mode():
    output = model(**training_inputs, labels=training_inputs["input_ids"])

output.loss

tensor(3.9355)

### 16.12.3 Autoregressive Generation

In [22]:
prompt = "Once upon a time"
inputs = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
inputs

tensor([[6403, 1980,  253,  655]])

In [27]:
max_new_tokens = 20

generated = inputs.clone()

for _ in range(max_new_tokens):
    with torch.inference_mode():
        outputs = model(input_ids=generated)
    # take the predicted logits for the last token in the current input
    # and select the most likely one
    next_token_logits = outputs.logits[:, -1, :]
    next_token_id = next_token_logits.argmax(dim=-1, keepdim=True)
    # appends the newly generated token to the sequence
    generated = torch.cat([generated, next_token_id], dim=-1)
    # if the next token is an EOS (end-of-sentence) token, stios
    if (next_token_id.item() == tokenizer.eos_token_id):
        break

In [28]:
tokenizer.decode(generated[0], skip_special_tokens=True)

'Once upon a time, there was a little girl named Lily. She lived in a big house with her family, but'

## 16.13 Choosing the Next Token

### 16.13.1 Greedy Decoding

In [29]:
next_token_id = next_token_logits.argmax(dim=-1, keepdim=True)

### 16.13.2 Sampling

In [ ]:
probs = torch.softmax(next_token_logits, dim=-1)
next_token_id = torch.multinomial(probs, num_samples=1)

In [31]:
n_samples = 3
max_new_tokens = 20

for _ in range(n_samples):
    generated = inputs.clone()
    for _ in range(max_new_tokens):
        with torch.inference_mode():
            outputs = model(input_ids=generated)
        # take the predicted logits for the last token in the current input
        # and samples the next token
        next_token_logits = outputs.logits[:, -1, :]
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token_id = torch.multinomial(probs, num_samples=1)
        # appends the newly generated token to the sequence
        generated = torch.cat([generated, next_token_id], dim=-1)
        # if the next token is an EOS (end-of-sentence) token, stios
        if (next_token_id.item() == tokenizer.eos_token_id):
            break
    
    print(tokenizer.decode(generated[0], skip_special_tokens=True))

Once upon a time in the land of Epirus, there was a person named Fulbert, who had twelve special helpers
Once upon a time in Lusanna, there lived a friendly little boy named Timmy. He grew up to be a curious
Once upon a time, deep in the heart of the Amazon rainforest, there were waybackers worried about ordering them gently


### 16.13.3 Temperature

We can modify the shape of the distribution before sampling using the temperature (T) argument. Given produced logits (z), the temperature (T) setting transforms them before applying softmax (to conver them into probabilities) using the expression:

$$
\Large
\frac{z}{T}
$$

Since softmax increases the skew of the distribution, higher values sharpen the distribution. Therefore, temperature values below 1.0 will make it sharper, so the highest-probability tokens become relatively more dominant, leaning towards deterministic behavior. On the other hand, temperatures above 1.0 will increasingly flatten the distribution, so more and more alternatives receive meaningful probability mass. An infinite temperature would effectively converge to a completely random uniform distribution. Temperature only matters when we're sampling. If we're always taking `argmax()`, dividing all logits by a positive temperature does not change which logit is largest.

In code, it is as simple as this:

In [ ]:
temperature = 0.8
probs = torch.softmax(next_token_logits / temperature, dim=-1)

In [37]:
temperatures = [0.001, 1.0, 1000.0]
max_new_tokens = 20

for temperature in temperatures:
    generated = inputs.clone()
    for _ in range(max_new_tokens):
        with torch.inference_mode():
            outputs = model(input_ids=generated)
        # take the predicted logits for the last token in the current input
        # and samples the next token
        next_token_logits = outputs.logits[:, -1, :]
        # we divide the logits by the temperature
        probs = torch.softmax(next_token_logits / temperature, dim=-1)
        next_token_id = torch.multinomial(probs, num_samples=1)
        # appends the newly generated token to the sequence
        generated = torch.cat([generated, next_token_id], dim=-1)
        # if the next token is an EOS (end-of-sentence) token, stios
        if (next_token_id.item() == tokenizer.eos_token_id):
            break
    
    print(tokenizer.decode(generated[0], skip_special_tokens=True))

Once upon a time, there was a little girl named Lily. She lived in a big house with her family, but
Once upon a time, Adi's family was moving from a tiny village near Marwari in Hejatpur
Once upon a time WyrorRather impoverished iconography homelessNumCreat gull Gautwide Monster clever lact cryptocurrencies percent monitoredalytic� concluding


### 16.13.4 Top-K/Top-P Sampling

Sampling from all 49,152 vocabulary tokens may occasionally select extremely unlikely tokens. **Top-k sampling** limits sampling to the (k) tokens with the largest scores. Everything outside that set receives zero sampling probability.

**Top-p**, or **nucleus sampling**, uses a dynamic number of tokens. Instead of specifying a given number of candidates exactly, we use a threshold for the probability (such as 0.9) and keep the smallest set of high-probability tokens whose cumulative probability reaches approximately 90%. For one prediction, that might require 12 tokens, while, for another, only 3. For a particularly uncertain distribution, it might require dozens. This adapts the candidate set to the shape of the model's prediction. We'll use these decoding parameters more extensively in the next chapter, where we work with instruction-tuned models. For now, the important point is: the model produces logits, but it is the decoding strategy that decides how those logits become the next token.

## 16.14 Using `generate()`

While our manual generation loop was quite simple, real generation needs to handle many additional details:

- batching;
- attention masks;
- padding;
- stopping criteria;
- end-of-sequence (EOS) tokens;
- different decoding strategies;
- caching;
- generation length;
- multiple returned sequences.

Transformers provides the `generate()` method that can handle all these details for us.

### 16.14.1 Greedy Generation

In [46]:
prompt = "Once upon a time"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

In [49]:
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

tokenizer.decode(output_ids[0], skip_special_tokens=True)

'Once upon a time, there was a little girl named Lily. She lived in a big house with her family, but'

### 16.14.2 Sampling

In [52]:
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

tokenizer.decode(output_ids[0], skip_special_tokens=True)

'Once upon a time over seventy people are sitting outside working long, very close periods while someone else walks about without anything except'

## 16.15 Efficient Autoregressive Inference

Our manual implementation has one rather serious inefficiency. Suppose our prompt has 100 tokens. To generate token 101, we process tokens from the first to the 100th. Then we append our prediction to the sequence. Now, to generate token 102, our naïve loop processes all tokens, from the first to the 101th again. And so on, and so forth. We keep recomputing representations for tokens we've already processed, and that's wasteful.

Remember what self-attention needs: queries, keys, and values. When generating a new token, its query is new, but the keys and values corresponding to previous tokens have already been computed. Instead of recreating them, we can save them: this saved state is the **KV cache**.

### 16.15.1 KV Cache

After processing the prompt, the model stores the key and value vectors associated with previous positions at every attention layer. When one new token arrives, the model computes the new token's representations and combines them with the cached keys and values. The first pass processes the full prompt and saves K and V, then, for everty next token, it computes Q, K, and V for the new token alone while reusing the old saved K and V. The model therefore avoids recomputing the complete history from scratch on every generation step. Hugging Face's current generation system uses cache objects specifically for this purpose; the default dynamic cache grows as generation proceeds.

### 16.15.2 The Memory Trade-Off

Caching saves computation, but it costs memory. The cache needs to store keys and values for:
- every cached token;
- every relevant attention layer;
- every key/value head.

As context length increases, the KV cache can become a significant part of inference memory usage. This is one reason grouped-query attention matters. Recall that SmolLM2 uses nine query heads but only three key/value heads. Fewer key/value heads mean fewer cached tensors. To address different aspects of the inference-memory problem, fifferent model architectures use techniques such as:
- grouped-query attention;
- multi-query attention;
- sliding-window attention;
- cache quantization;

Current Transformers also supports several cache strategies that trade memory, compilation compatibility, and speed differently. We don't need to configure them manually here, as `generate()` handles normal caching for us.

### 16.15.3 Context Length

The KV cache also helps illustrate what a **context window** means operationally. At any generation step, the model can condition its prediction on the tokens included in its available context. Our SmolLM2 configuration specifies:

In [53]:
model.config.max_position_embeddings

8192

## 16.16 Base Language Models

In [ ]:
prompt = "Explain why the sky is blue."

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Explain why the sky is blue.

The answer is that the sky is blue because of the way light is scattered by molecules in the air. When light passes through a prism, it is broken up into a series of colors. The colors are separated by the prism and are then


During pretraining, the basic objective was to predict the next token, given the previous tokens. The training data may contain:

- articles;
- discussions;
- code;
- educational text;
- stories;
- documentation;
- many other forms of text.

By learning to predict tokens across enormous collections, the model learns useful patterns about language, syntax, structure, factual associations, code, and relationships between concepts. But nothing in the basic causal-language-modeling objective requires the model to answer helpfully whenever the user asks something.

A base model interprets its input primarily as a sequence to continue. The model might continue in the desired format because similar patterns occurred during pretraining, but that behavior isn't the same thing as having been explicitly trained to follow instructions. While the underlying architecture may remain the same, the model has to be further trained to respond effectively to requests, instead of simply auto-completing text.

## 16.17 From Base Models to Instruction Models

The Transformer decoder architecture we've studied is already the core of an LLM. It contains:

- token embeddings;
- positional information such as RoPE;
- causal self-attention;
- feed-forward networks;
- many stacked decoder blocks;
- a language-modeling head;
- next-token prediction.

Nothing in the list requires a chat interface. An instruction model starts from a pretrained language model and receives additional training on examples that demonstrate desired behaviors. For example:

```
<|im_start|>User:
Summarize this paragraph.<|im_end|>

<|im_start|>Assistant:
...
...<|im_end|>
```

This process teaches the model how text representing a **request** should relate to text representing a desirable **response**. Conversational models also need a consistent way to distinguish roles such as `system`, `user`, and `assistant`. Those roles are ultimately encoded as tokens too. Different model families use different control tokens and formatting conventions. That's why modern chat models use **chat templates**. But those belong to the next chapter.

## 16.18 Looking Ahead

Let's put everything together. In Chapter 15, we focused on encoder models and representations. Text was converted into embeddings so we could compare meaning and retrieve related documents. This chapter approached Transformers from the generative side.

Self-attention begins with learned transformations of token representations into queries, keys, and avlues, and computes attention scores. Multiple attention heads allow several attention patterns to operate in parallel, while modern techniques such as grouped-query attention reduce the number of key/value representations that need to be stored. Attention alone does not know the ordering of tokens, so the model also needs positional information. SmolLM2 uses **Rotary Position Embeddings**, which rotate query and key vectors according to their sequence positions. To turn attention into an autoregressive language model, we impose a causal constraint, so each token can attend only to itself and previous positions.

A decoder-only Transformer repeatedly applies these operations and eventually maps every hidden state into vocabulary-sized logits. During inference, the final next-token prediction is selected, appended to the sequence, and fed back into the model, that's autoregressive generation. Decoding strategies such as greedy decoding, temperature, top-k, and top-p determine how we choose among the model's possible next tokens. A KV cache makes generation more efficient by reusing key and value representations from earlier tokens rather than recomputing them from scratch.

Finally, we saw an important distinction. The model we used was a **base language model**. It knows how to continue text but that doesn't automatically make it an assistant. In the next chapter, **Working with Instruction-Tuned LLMs**, we'll move from text continuation to instruction following and and learn how modern LLM applications use:
- base and instruction-tuned checkpoints;
- user, assistant, and system roles;
- chat templates;
- generation controls;
- context windows;
- practical model loading and memory considerations;
- high-level inference interfaces.

The architecture won't suddenly become a different kind of model, we'll still be generating one token at a time, but the model will be trained to handle these tokens differently.